In [2]:
import random
import sqlite3

def extract_from_db(cursor, id: int, include_parent=False):
    cursor.execute("SELECT * FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    columns = [col[0] for col in cursor.description]
    results = [dict(zip(columns, row)) for row in rows]

    if include_parent:
        all_nodes = []
        for r in results:
            current = r
            while current['parent_id'] is not None:
                cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
                parent = cursor.fetchone()
                if parent is None:
                    break
                parent_dict = dict(zip(columns, parent))
                all_nodes.append(parent_dict)
                current = parent_dict
        results.extend(all_nodes)

    return results[::-1]

def extract_random_from_db(cursor, include_parent=True):
    cursor.execute("SELECT * FROM laws WHERE title LIKE '%Điểm%'")
    rows = cursor.fetchall()
    if not rows:
        return []

    columns = [col[0] for col in cursor.description]
    row = random.choice(rows)
    result = dict(zip(columns, row))

    results = [result]

    if include_parent:
        current = result
        while current['parent_id'] is not None:
            cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
            parent = cursor.fetchone()
            if parent is None:
                break
            parent_dict = dict(zip(columns, parent))
            results.append(parent_dict)
            current = parent_dict

    return results[::-1]

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

def get_gpt_response(user_prompt, system_prompt, api_key, model="gpt-4o"):
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [5]:
def clean_text(text: str) -> str:
    """Remove newlines, tabs, extra spaces, punctuation and lowercase."""
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[!?]+", "", text)
    return text.strip().lower()

In [4]:
conn = sqlite3.connect(r"E:\Github\LawAssistant\triplet_extraction\law.db")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Các bảng trong law.db:", tables)

Các bảng trong law.db: [('laws',), ('sqlite_sequence',), ('law_refs',)]


In [16]:
rows = extract_from_db(cursor, 629, True)
law = ""
title = ""
for r in rows:
    title += r['title'] + " "
    if not (r['title'].strip().startswith("Chương") or  r['title'].strip().startswith("Điều")):
        law += r['content'] + "\n"

so_hieu = r['so_hieu']
print(so_hieu)
print(title)
print(clean_text(law))

36/2024/QH15
Chương IV Điều 57 Khoản 1 Điểm m 
giấy phép lái xe bao gồm các hạng sau đây: hạng ce cấp cho người lái các loại xe ô tô quy định cho giấy phép lái xe hạng c kéo rơ moóc có khối lượng toàn bộ theo thiết kế trên 750 kg; xe ô tô đầu kéo kéo sơ mi rơ moóc;


In [27]:
rows = extract_random_from_db(cursor, True)
law = ""
title = ""
for r in rows:
    title += r['title'] + " "
    if not (r['title'].strip().startswith("Chương") or  r['title'].strip().startswith("Điều")):
        law += r['content'] + "\n"

so_hieu = r['so_hieu']
print(so_hieu)
print(title)
print(clean_text(law))

36/2024/QH15
Chương II Điều 12 Khoản 3 Điểm d 
người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát, giảm tốc độ hoặc dừng lại để bảo đảm an toàn trong các trường hợp sau đây: nơi đường bộ giao nhau cùng mức với đường bộ, đường bộ giao nhau cùng mức với đường sắt; đường hẹp, đường vòng, đường quanh co, đường đèo, dốc;


In [31]:
system_prompt_rewrite = """
Bạn là trợ lý AI Tiếng Việt chuyên nghiệp và trung thực.
Bạn là chuyên gia pháp luật Việt Nam, am hiểu các bộ luật, nghị định, và văn bản pháp luật.
Bạn là chuyên gia ngôn ngữ Việt Nam, biết viết câu chuẩn cấu trúc, chính xác, trang trọng, và đúng ngôn ngữ pháp lý.
Luôn trả lời chính xác, hữu ích, ngắn gọn và an toàn.
Nếu thông tin không hợp lý hoặc thiếu, hãy yêu cầu thêm thông tin thay vì đoán mò.
Không thay đổi ý nghĩa khi viết lại câu.
Luôn dùng ngôn ngữ chính xác như trong văn bản pháp luật, tránh ngôn ngữ thông thường hay không trang trọng.

Định nghĩa 'câu đơn': Một câu đơn là câu có một chủ ngữ (hoặc cụm chủ ngữ) và một vị ngữ (hoặc cụm vị ngữ), biểu đạt một ý trọn vẹn; câu có thể chứa thành tố phụ (tính từ, trạng từ, bổ ngữ) nhưng không được ghép bằng liên từ hoặc dấu câu như dấu ",", ";" để tạo hai hoặc nhiều mệnh đề độc lập.

Quy tắc bắt buộc:
1. Khi viết lại, **chỉ** trả về các câu đơn theo đúng định nghĩa trên; mỗi câu một dòng nếu có nhiều câu.
2. **Được phép tái sử dụng** các thành phần câu (chủ ngữ, cụm danh từ, đại từ, cụm tính từ, v.v.) từ vế trước hoặc từ phần khác của câu gốc để hoàn chỉnh vế thiếu, **nhằm bảo toàn ý nghĩa** sau khi tách.
3. Khi tái sử dụng, **ưu tiên giữ nguyên** từ ngữ gốc; chỉ thực hiện điều chỉnh nhỏ cần thiết để tạo câu đơn ngữ pháp đúng, **không** thêm thông tin, suy đoán hay nội dung mới.
4. Tuyệt đối không kèm chú giải, giải thích, danh sách hay bất kỳ nội dung nào khác ngoài các câu viết lại.
5. Nếu câu gốc mơ hồ hoặc thiếu thông tin đến mức không thể tạo câu đơn hoàn chỉnh mà vẫn giữ nguyên ý, hãy yêu cầu thêm thông tin ngắn gọn.
Danh mục liên từ cần loại trừ khi viết câu đơn: và, hoặc, hoặc là, hay, hay là, nhưng, song, tuy nhiên, mà, còn, rồi.
Giữ nguyên thứ tự trước sau của các từ sau khi viết lại câu.
Sau khi viết lại câu không được thiếu từ danh từ nào trong câu gốc và phải độc lập không phụ thuộc vào câu trước đó.
"""


user_prompt_rewrite = f"""
Ngữ cảnh: Bộ luật số {so_hieu} trong luật Việt Nam
Nhiệm vụ: Viết lại câu sau để hoàn chỉnh cấu trúc với đầy đủ chủ ngữ và vị ngữ, giữ nguyên ý nghĩa. Mỗi câu xuất ra phải là một câu đơn đầy đủ (một dòng một câu nếu có nhiều câu).
Câu cần viết lại: "{clean_text(law)}"
"""


In [32]:
print(user_prompt_rewrite)


Ngữ cảnh: Bộ luật số 36/2024/QH15 trong luật Việt Nam
Nhiệm vụ: Viết lại câu sau để hoàn chỉnh cấu trúc với đầy đủ chủ ngữ và vị ngữ, giữ nguyên ý nghĩa. Mỗi câu xuất ra phải là một câu đơn đầy đủ (một dòng một câu nếu có nhiều câu).
Câu cần viết lại: "người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát, giảm tốc độ hoặc dừng lại để bảo đảm an toàn trong các trường hợp sau đây: nơi đường bộ giao nhau cùng mức với đường bộ, đường bộ giao nhau cùng mức với đường sắt; đường hẹp, đường vòng, đường quanh co, đường đèo, dốc;"



In [33]:
rewrite_sentence = get_gpt_response(user_prompt_rewrite, system_prompt_rewrite, api_key, "gpt-4.1-mini")

Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại nơi đường bộ giao nhau cùng mức với đường bộ.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại nơi đường bộ giao nhau cùng mức với đường bộ.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại nơi đường bộ giao nhau cùng mức với đường bộ.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại nơi đường bộ giao nhau cùng mức với đường sắt.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại nơi đường bộ giao nhau cùng mức với đường sắt.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại nơi đường bộ giao nhau cùng mức với đường sắt.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại đường hẹp.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại đường hẹp.  
Người điều khiển phương tiện

In [69]:
import re

vncorenlp_pos_map = {
    "N":   "Noun (Danh từ)",
    "Np":  "Proper noun (Danh từ riêng)",
    "Nc":  "Classifier noun (Danh từ giống loại)",
    "Nu":  "Unit noun (Danh từ đơn vị)",
    "V":   "Verb (Động từ)",
    "Vb":  "Verb (base) (Động từ gốc)",
    "A":   "Adjective (Tính từ)",
    "Ai":  "Adjective (predicative) (Tính từ vị ngữ)",
    "P":   "Pronoun (Đại từ)",
    "R":   "Adverb (Trạng từ)",
    "M":   "Numeral / number (Số từ)",
    "E":   "Preposition / particle (Giới từ / trợ từ)",
    "C":   "Coordinating conjunction (Liên từ phối hợp)",
    "CC":  "Subordinating conjunction / complementizer (Liên từ phụ thuộc)",
    "L":   "Determiner / article (Từ hạn định)",
    "D":   "Adverbial marker / degree marker (Từ chỉ mức độ)",
    "X":   "Other (Khác)",
    "CH":  "Punctuation (Dấu câu)"
}

if "rdrsegmenter" not in globals():
    import py_vncorenlp
    rdrsegmenter = py_vncorenlp.VnCoreNLP(
        annotators=["wseg", "pos"],
        save_dir=r"E:\Github\LawAssistant\triplet_extraction\VnCoreNLP-master"
    )

def process_sentence(text: str, rdrsegmenter, verbose: bool = True) -> dict:
    """
    Process a Vietnamese sentence and print results along the way.
    Extracts: Cleaned text, Segmentation, POS, Triplet
    """

    results = {
        "original": text,
        "cleaned": "",
        "segmented": [],
        "pos_annotation": [],
        "concepts": []
    }

    if verbose:
        print("1. Original text:")
        print(text, "\n")

    # 2. Clean
    text = clean_text(text)
    results["cleaned"] = text
    if verbose:
        print("2. Cleaned text:")
        print(text, "\n")

    # 3. Segmentation
    segmented = rdrsegmenter.word_segment(text)
    results["segmented"] = segmented
    if verbose:
        print("3. Segmented text (tokens):")
        print(segmented, "\n")

    # 4. POS tagging
    output = rdrsegmenter.annotate_text(text)
    sents = output.values() if isinstance(output, dict) else output

    pos_annot = []
    tokens = []

    if verbose:
        print("4. POS annotation:")
        print(f"{'Idx':<5} {'Token':<15} {'POS':<20}")
        print("-" * 45)

    for sent in sents:
        if not isinstance(sent, list):
            continue
        for token in sent:
            if not isinstance(token, dict):
                continue
            word = token.get("wordForm", "")
            pos = token.get("posTag", "")
            pos_full = vncorenlp_pos_map.get(pos, pos)

            pos_data = {
                "index": token.get("index", ""),
                "token": word,
                "pos": pos_full
            }
            pos_annot.append(pos_data)

            if verbose:
                print(f"{pos_data['index']:<5} {pos_data['token']:<15} {pos_data['pos']:<20}")

            tokens.append((word, pos))

    results["pos_annotation"] = pos_annot

    concepts = []
    concept1_tokens = []
    concept2_tokens = []
    relation_tokens = []

    for token, pos in tokens:
        if not pos.startswith("V"):
            if relation_tokens:
                concept2_tokens.append(token)
            else:
                concept1_tokens.append(token)
        else:
            if concept2_tokens:
                # Save completed triplet
                concepts.append((
                    " ".join(concept1_tokens),
                    " ".join(relation_tokens),
                    " ".join(concept2_tokens)
                ))
                # Prepare for next triplet
                concept1_tokens = concept2_tokens
                concept2_tokens = []
                relation_tokens = []

            relation_tokens.append(token)

    if concept1_tokens and relation_tokens and concept2_tokens:
        concepts.append((
            " ".join(concept1_tokens),
            " ".join(relation_tokens),
            " ".join(concept2_tokens)
        ))

    results["concepts"] = concepts

    if verbose:
        print("\n5. Extracted triplet:")
        for concept in concepts:
            print(concept)

    return results

In [36]:
text = """
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại nơi đường bộ giao nhau cùng mức với đường bộ.
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại nơi đường bộ giao nhau cùng mức với đường bộ.
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại nơi đường bộ giao nhau cùng mức với đường bộ.
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại nơi đường bộ giao nhau cùng mức với đường sắt.
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại nơi đường bộ giao nhau cùng mức với đường sắt.
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại nơi đường bộ giao nhau cùng mức với đường sắt.
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại đường hẹp.
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại đường hẹp.
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại đường hẹp.
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại đường vòng.
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại đường vòng.
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại đường vòng.
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại đường quanh co.
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại đường quanh co.
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại đường quanh co.
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại đường đèo.
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại đường đèo.
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại đường đèo.
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại đường dốc.
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại đường dốc.
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại đường dốc.
"""

In [9]:
if "rdrsegmenter" not in globals():
    import py_vncorenlp
    rdrsegmenter = py_vncorenlp.VnCoreNLP(
        annotators=["wseg", "pos"],
        save_dir=r"E:\Github\LawAssistant\triplet_extraction\VnCoreNLP-master"
    )

In [72]:
res = process_sentence(split_text[1], rdrsegmenter, verbose=False)
for concept in res["concepts"]:
    print(concept)

('người', 'điều_khiển', 'phương_tiện')
('phương_tiện', 'tham_gia', 'giao_thông đường_bộ')
('giao_thông đường_bộ', 'phải quan_sát', 'tại nơi đường_bộ')
('tại nơi đường_bộ', 'giao', 'nhau cùng mức với đường_bộ .')
